# Building Height Prediction

Phase 2 of [3D_plan1.md](../docs/3D_plan1.md): predict per-pixel building heights
from satellite imagery using CNN.

**Two modes**
| Mode | Description |
|------|-------------|
| `pretrained` | Depth Anything V2 Small (HuggingFace, zero-shot). Calibrated to metres using OSM/Phase-1 heights in the same tile. No training needed. |
| `unet` | Trained U-Net checkpoint (see `Train_Height_CNN.ipynb`). Best accuracy when tile coverage exists. |

**Prerequisites**
```bash
pip install transformers torch torchvision timm
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

from app.session.terrain_session import TerrainSession

## 1. Create a session and fetch satellite imagery

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Configuration — change bbox, zoom, or model below.
# ──────────────────────────────────────────────────────────────────────────────

BBOX = dict(north=41.395, south=41.375, east=2.175, west=2.145)  # Barcelona Eixample
ZOOM = 17
MODEL = "pretrained"    # "pretrained" or "unet"
DEVICE = "cpu"          # "cpu" or "cuda"
UNET_CHECKPOINT = None  # Path to .pt file when MODEL="unet"; None → default

# ──────────────────────────────────────────────────────────────────────────────

s = TerrainSession(bbox=BBOX)
print(f"Session created: {BBOX}")

In [ ]:
# Fetch satellite RGB
s.fetch_satellite(zoom=ZOOM)
print("Satellite imagery fetched")

## 2. (Optional) Fetch Phase-1 building heights for calibration

When `model="pretrained"` the DA2 relative depth is calibrated to absolute
metres using any OSM / provider heights you have loaded.  The more coverage
the better, but this step is optional.

In [ ]:
PROVIDERS = ["wsf3d", "ndsm"]  # change as needed

try:
    s.fetch_building_heights(providers=PROVIDERS)
    print("Phase-1 heights fetched — will be used for calibration")
except Exception as e:
    print(f"Skipping Phase-1 heights ({e})")

## 3. Predict heights

In [ ]:
s.predict_heights(
    model=MODEL,
    checkpoint=UNET_CHECKPOINT,
    device=DEVICE,
)
result = s.predicted_heights
print(f"source  : {result.source_name}")
print(f"shape   : {result.raster.shape}")
print(f"range   : [{np.nanmin(result.raster):.1f}, {np.nanmax(result.raster):.1f}] m")
print(f"mean    : {np.nanmean(result.raster):.1f} m")
print(f"conf    : [{np.nanmin(result.confidence):.2f}, {np.nanmax(result.confidence):.2f}]")

## 4. Visualise

In [ ]:
import base64, io
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# — Satellite RGB ————————————————————————————————————————
sat_b64 = s.satellite.get("image_b64") or s.satellite.get("data")
sat_rgb = np.array(Image.open(io.BytesIO(base64.b64decode(sat_b64))).convert("RGB"))
axes[0].imshow(sat_rgb)
axes[0].set_title("Satellite RGB")
axes[0].axis("off")

# — Predicted heights ————————————————————————————————————
im1 = axes[1].imshow(result.raster, cmap="hot", vmin=0, vmax=60)
axes[1].set_title(f"Predicted Heights ({result.source_name})")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], label="metres")

# — Confidence ——————————————————————————————————————————
im2 = axes[2].imshow(result.confidence, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("Confidence")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.savefig("../output/height_prediction_preview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → output/height_prediction_preview.png")

## 5. Compare DA2 vs Phase-1 heights (optional)

In [ ]:
if hasattr(s, "building_heights") and s.building_heights is not None:
    phase1 = s.building_heights.raster
    valid = ~np.isnan(phase1)
    if valid.any():
        pred_at_known = result.raster[valid]
        true_at_known = phase1[valid]
        mae = np.mean(np.abs(pred_at_known - true_at_known))
        print(f"MAE vs Phase-1 heights: {mae:.2f} m  (n={int(valid.sum())} pixels)")

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(true_at_known, pred_at_known, alpha=0.3, s=3, c="steelblue")
        lim = max(true_at_known.max(), pred_at_known.max()) * 1.05
        ax.plot([0, lim], [0, lim], "r--", lw=1)
        ax.set_xlabel("Phase-1 height (m)")
        ax.set_ylabel("Predicted height (m)")
        ax.set_title(f"Prediction vs Phase-1  (MAE={mae:.2f} m)")
        plt.tight_layout()
        plt.show()
    else:
        print("No valid Phase-1 pixels to compare")
else:
    print("Phase-1 heights not loaded — skipping comparison")

## 6. Merge predicted heights into buildings GeoDataFrame (optional)

If you have OSM buildings fetched, the predicted height raster can be used
to fill missing building heights.

In [ ]:
if hasattr(s, "buildings") and s.buildings is not None:
    from app.server.core.osm import enhance_buildings_with_raster

    bbox_tuple = (BBOX["north"], BBOX["south"], BBOX["east"], BBOX["west"])
    enhanced = enhance_buildings_with_raster(
        s.buildings,
        result.raster,
        bbox_tuple,
        confidence_raster=result.confidence,
        min_confidence=0.3,
    )
    print(f"Buildings with predicted heights: {len(enhanced)}")
else:
    print("No buildings loaded — call s.fetch_buildings() first if needed")